In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq
api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=api_key,model="Gemma-7b-It")
llm

ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x0000021FBD6968D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021FBD80E2D0>, model_name='Gemma-7b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

In [4]:
speech = """The 2024 Summer Olympics, officially known as the Games of the XXXIII Olympiad, is an international multi-sport event that is scheduled to take place from July 26 to August 11, 2024, in Paris, France. This will be the third time that Paris has hosted the Olympics, having previously hosted the games in 1900 and 1924. The event will feature a wide range of sports and will bring together athletes from around the world to compete for medals and represent their countries on the global stage."""

In [5]:
speech

'The 2024 Summer Olympics, officially known as the Games of the XXXIII Olympiad, is an international multi-sport event that is scheduled to take place from July 26 to August 11, 2024, in Paris, France. This will be the third time that Paris has hosted the Olympics, having previously hosted the games in 1900 and 1924. The event will feature a wide range of sports and will bring together athletes from around the world to compete for medals and represent their countries on the global stage.'

In [19]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

chat_messages = [
    SystemMessage(content="You are an expert assistant with expertise in summarizing speeches"),
    HumanMessage(content=f"Please summarize:\n{speech}")
]

llm = ChatOpenAI(
    model="gpt-4o-mini",   # better & current model
    groq_api_key="api_key"
)

response = llm.invoke(chat_messages)

print(response.content)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [13]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
num_tokens = len(enc.encode(speech))

print(num_tokens)

108


In [14]:
response = llm.invoke(chat_messages)
print(response.content)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: gsk_yxY6********************************************13oC. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

Prompt Template TEXT Summerization 


In [29]:
from langchain_core.prompts import PromptTemplate
from langchain.chains.llm import LLMChain
template = """You are a helpful assistant that summarizes the text.
{text}
"""
prompt = PromptTemplate.from_template(template)
chain = LLMChain(llm=llm, prompt=prompt)


ModuleNotFoundError: No module named 'langchain.chains'

In [22]:
prompt.format(speech = speech, language = "english")

NameError: name 'prompt' is not defined

In [31]:
complete_prompt = prompt.format(text = speech)
print(complete_prompt)

NameError: name 'prompt' is not defined

In [ ]:
llm.get_num_tokens(complete_prompt)
response = llm.invoke(complete_prompt)
print(response.content)

In [ ]:
llm_chain  = LLMChain(llm=llm, prompt=prompt)
response = llm_chain.run(text = speech)
print(response)

StuffDocumentChain Text Summarization

In [ ]:
from PyPDF2 import PdfReader


In [ ]:
# provide the path of  pdf file/files.
pdfreader = PdfReader('apjspeech.pdf')

In [ ]:
from typing_extensions import Concatenate
# read text from pdf
text = ''
for i, page in enumerate(pdfreader.pages):
    content = page.extract_text()
    if content:
        text += content

In [ ]:
text

In [ ]:
llm = ChatOpenAI(temperature=0, model_name='gpt-3.5-turbo')

In [ ]:
from langchain import PromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.chains.summarize import load_summarize_chain
from langchain.docstore.document import Document

In [ ]:
template = '''Write a concise and short summary of the following speech.
Speech: `{text}`
'''
prompt = PromptTemplate(
    input_variables=['text'],
    template=template
)

In [ ]:
chain = load_summarize_chain(
    llm,
    chain_type='stuff',
    prompt=prompt,
    verbose=False
)
output_summary = chain.run(docs)

In [ ]:
output_summary

Summarizing Large Documents Using Map Reduce

In [ ]:
from langchain import PromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.chains.summarize import load_summarize_chain
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
# provide the path of  pdf file/files.
pdfreader = PdfReader('apjspeech.pdf')
from typing_extensions import Concatenate
# read text from pdf
text = ''
for i, page in enumerate(pdfreader.pages):
    content = page.extract_text()
    if content:
        text += content

In [ ]:
llm = ChatOpenAI(temperature=0, model_name='gpt-3.5-turbo')

In [ ]:
llm.get_num_tokens(text)

In [ ]:
## Splittting the text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=20)
chunks = text_splitter.create_documents([text])

In [ ]:
len(chunks)

In [ ]:
chain = load_summarize_chain(
    llm,
    chain_type='map_reduce',
    verbose=False
)
summary = chain.run(chunks)

In [ ]:
summary

Map Reduce With Custom Prompts

In [ ]:
chunks_prompt="""
Please summarize the below speech:
Speech:`{text}'
Summary:
"""
map_prompt_template=PromptTemplate(input_variables=['text'],
                                    template=chunks_prompt)

In [ ]:

final_combine_prompt='''
Provide a final summary of the entire speech with these important points.
Add a Generic Motivational Title,
Start the precise summary with an introduction and provide the
summary in number points for the speech.
Speech: `{text}`
'''
final_combine_prompt_template=PromptTemplate(input_variables=['text'],
                                             template=final_combine_prompt)

In [ ]:
summary_chain = load_summarize_chain(
    llm=llm,
    chain_type='map_reduce',
    map_prompt=map_prompt_template,
    combine_prompt=final_combine_prompt_template,
    verbose=False
)
output = summary_chain.run(chunks)

In [ ]:
output


RefineChain For Summarization

In [ ]:
chain = load_summarize_chain(
    llm=llm,
    chain_type='refine',
    verbose=True
)
output_summary = chain.run(chunks)

In [ ]:
output_summary